# Part 0: Prerequisites
## Import neccessary libraries

In [1]:
import sys
sys.path.append('../src')

import numpy as np
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# plot and ML model
from sixth_meeting.visualizer import TransitionPointsVisualizer
from fifth_meeting.create_dataset import HammingDecayPredictor

print("All imports successful!")

All imports successful!


## Implement exponential decay function and a way to fit it to our data

In [2]:
# Fit hyperbolic decay curves to the smoothed data ((A / B * x) + C, A - initial value, B - decay rate, C - asymptotic value (bias))
def hyperbolic_decay(x, A, B, C):
    return (A / (B * x)) + C

def exponential_decay(x, A, B, C):
    return A * np.exp(-B * x) + C

def fit_curve_to_data(x_data, y_data, poly_type=exponential_decay):

    # Initial A, B, C guesses
    C_guess = np.mean(y_data[-5:])  # Estimate offset from last few points
    A_guess = np.max(y_data) - C_guess  # Amplitude from peak minus offset

    non_zero_mask = y_data - C_guess > 0.01
    if np.sum(non_zero_mask) > 2:
        log_y = np.log(y_data[non_zero_mask] - C_guess)
        x_for_fit = x_data[non_zero_mask]
        B_guess = -np.polyfit(x_for_fit, log_y, 1)[0]
    else:
        B_guess = 0.1  # Default guess

    initial_guess = [A_guess, B_guess, C_guess]

    # Apply Nonlinear Least Squares fitting for A, B, C
    try:
        # Basic fit
        params_opt, params_cov = curve_fit(
            poly_type, 
            x_data, 
            y_data,
            p0=initial_guess,
            maxfev=10000  # Increase max function evaluations
        )
        
        A_fit, B_fit, C_fit = params_opt
        perr = np.sqrt(np.diag(params_cov))  # Parameter uncertainties
        
    except RuntimeError as e:
        print(f"Optimization failed: {e}")
        print("Trying with bounds...")
        
        # Try with bounds to help convergence
        params_opt, params_cov = curve_fit(
            poly_type,
            x_data,
            y_data,
            p0=initial_guess,
            bounds=([0, 0, -np.inf], [np.inf, np.inf, np.inf]),  # A,B ≥ 0
            maxfev=10000
        )
        
        A_fit, B_fit, C_fit = params_opt

    # return predicted values and parameters
    return A_fit, B_fit, C_fit

## Load raw simulation data of the test circuits to display along with our Hellinger prediction

In [3]:
loaded = np.load(f'../data/sixth_meeting/test_circuits_simulation_dataset.npz', allow_pickle=True)
names_dataset = loaded['name']
circuits_info_dataset = loaded['circuit_info']
shots_dataset = loaded['shots_range']
noise_dataset = loaded['noise_range']
hamming_dataset = loaded['hamming_outputs']
hellinger_dataset = loaded['hellinger_outputs']

#--- Smooth the Hellinger and Hamming data using Savitzky-Golay filter
smoothed_hellinger = np.array([
            [savgol_filter(savgol_filter(hellinger_dataset[i][j], 11, 2), 11, 2)
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

smoothed_hamming = np.array([
            [[savgol_filter(savgol_filter(hamming_dataset[i][j][k], 11, 2), 11, 2)
              for k in range(len(noise_dataset[i]))]
             for j in range(10)]
            for i in range(len(names_dataset))
        ])

#--- Calculate the standard deviation of the smoothed Hamming data across the 10 runs
smoothed_hamming_std = [np.std(hamming_data, axis=0) for hamming_data in smoothed_hamming]


#--- Fit the smoothed Hellinger and Hamming data to the hyperbolic decay curve to extract A, B, C parameters
poly_hellinger_ABC = np.array([
                [fit_curve_to_data(shots_dataset[i], smoothed_hellinger[i][j])
                for j in range(len(noise_dataset[i]))]
                for i in range(len(names_dataset))
            ])

poly_hamming_std_ABC = np.array([
                [fit_curve_to_data(shots_dataset[i], smoothed_hamming_std[i][j])
                for j in range(len(noise_dataset[i]))]
                for i in range(len(names_dataset))
            ])

#--- Use the fitted A, B, C parameters to generate the fitted curves for Hellinger and Hamming data
poly_hellinger = np.array([
            [exponential_decay(shots_dataset[i], *poly_hellinger_ABC[i][j])
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

poly_hamming_std = np.array([
            [exponential_decay(shots_dataset[i], *poly_hamming_std_ABC[i][j])
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

# Part 1: Training our model
## Load training circuits data, normalize it, Create DataLoader

In [4]:
# load the training data
loaded_data = np.load('../data/sixth_meeting/ml_train_hellinger_dataset.npz') # 90 circuits dataset
X_raw, Y_raw = loaded_data['X'], loaded_data['Y']

scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y_raw)

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_raw, dtype=torch.float32)     # [hamming_std on 50 and 70 shots, circuit supermarq features]
Y_tensor = torch.tensor(Y_scaled, dtype=torch.float32)  # [fitted A, B, C parameters for the Hellinger decay curve]


# Create DataLoader
dataset = TensorDataset(X_tensor, Y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

## Train the model

In [5]:
### Initialize the HammingDecayPredictor model and train it
model = HammingDecayPredictor(input_dim=6, output_dim=3)

# MSE Loss is standard for regression. 
# Huber loss is less sensitive to outliers in data than MSE loss.
#criterion = nn.MSELoss()
criterion = nn.HuberLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

#--- Training loop
epochs = 1000
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_Y in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(batch_X)
        
        # Compute loss
        loss = criterion(predictions, batch_Y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_X.size(0)
        
    total_epoch_loss = epoch_loss / len(train_loader.dataset)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_epoch_loss:.4f}")

Epoch [10/1000], Loss: 0.4038
Epoch [20/1000], Loss: 0.3918
Epoch [30/1000], Loss: 0.3762
Epoch [40/1000], Loss: 0.3551
Epoch [50/1000], Loss: 0.3268
Epoch [60/1000], Loss: 0.2913
Epoch [70/1000], Loss: 0.2557
Epoch [80/1000], Loss: 0.2233
Epoch [90/1000], Loss: 0.2045
Epoch [100/1000], Loss: 0.1971
Epoch [110/1000], Loss: 0.1948
Epoch [120/1000], Loss: 0.1930
Epoch [130/1000], Loss: 0.1921
Epoch [140/1000], Loss: 0.1916
Epoch [150/1000], Loss: 0.1912
Epoch [160/1000], Loss: 0.1903
Epoch [170/1000], Loss: 0.1901
Epoch [180/1000], Loss: 0.1896
Epoch [190/1000], Loss: 0.1894
Epoch [200/1000], Loss: 0.1888
Epoch [210/1000], Loss: 0.1884
Epoch [220/1000], Loss: 0.1881
Epoch [230/1000], Loss: 0.1877
Epoch [240/1000], Loss: 0.1876
Epoch [250/1000], Loss: 0.1871
Epoch [260/1000], Loss: 0.1867
Epoch [270/1000], Loss: 0.1863
Epoch [280/1000], Loss: 0.1866
Epoch [290/1000], Loss: 0.1861
Epoch [300/1000], Loss: 0.1853
Epoch [310/1000], Loss: 0.1851
Epoch [320/1000], Loss: 0.1843
Epoch [330/1000],

# Part 2: Perform Hellinger decay parameters prediction on unseen circuits
## Load the test circuits data and do a prediction for each of them

In [ ]:
# load the test data
loaded_data = np.load('../data/sixth_meeting/test_ml_hellinger_dataset.npz')    # 18 circuits dataset
X_raw, _ = loaded_data['X'], loaded_data['Y']                                   # We don't need the Y values for testing, as we are predicting them

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_raw, dtype=torch.float32)

#--- Predict Hamming A, B, C parameters
model.eval()
with torch.no_grad():
    Y_pred_scaled = model(X_tensor)
    Y_pred = scaler_Y.inverse_transform(Y_pred_scaled.numpy())

# print(Y_pred.shape)

Y_reshaped = Y_pred.reshape(18, 1, 3)

## Visualize The Hamming STD, Actual Hellinger and its exponential fit, Predicted Hellinger and its exponential fit and display their saturation points

In [7]:
slope_range = np.arange(0.00012, 0.0005, 0.00003) # a range of different slope thresholds

hellinger_points = []
predicted_hellinger_points = []

#--- Calculate the saturation points for every circuit, every noise and slope level in the specified range, and store them in a list to be plotted later
for circuit_idx in range(len(names_dataset)):
    hellinger_points_circuit = []
    predicted_hellinger_points_circuit = []
    for noise_idx in range(len(noise_dataset[circuit_idx])):
        hellinger_points_noise = []
        predicted_hellinger_points_noise = []
        for slope_threshold in slope_range:
            A, B, _ = poly_hellinger_ABC[circuit_idx][noise_idx]
            point_idx = np.where(np.abs(-A * B * np.exp(-B * shots_dataset[circuit_idx])) < slope_threshold)[0]
            hellinger_points_noise.append(point_idx[0])

            A_pred, B_pred, _ = Y_reshaped[circuit_idx * len(noise_dataset[circuit_idx]) + noise_idx][0]
            point_idx_pred = np.where(np.abs(-A_pred * B_pred * np.exp(-B_pred * shots_dataset[circuit_idx])) < slope_threshold)[0]
            predicted_hellinger_points_noise.append(point_idx_pred[0])
        hellinger_points_circuit.append(hellinger_points_noise)
        predicted_hellinger_points_circuit.append(predicted_hellinger_points_noise)
    hellinger_points.append(hellinger_points_circuit)
    predicted_hellinger_points.append(predicted_hellinger_points_circuit)


#--- Calculate the predicted Hellinger curves using the predicted A, B, C parameters from the ML model
poly_predicted_hellinger = np.array([
            [exponential_decay(shots_dataset[i], *Y_reshaped[i][j])
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

#--- Visualize the results using the TransitionPointsVisualizer
exponential_decay_arrays = (poly_hellinger, poly_hamming_std, poly_predicted_hellinger)
saturation_points = (hellinger_points, predicted_hellinger_points)
visualizer_pred_hellinger = TransitionPointsVisualizer(names_dataset, shots_dataset, noise_dataset, smoothed_hellinger, smoothed_hamming_std, exponential_decay=exponential_decay_arrays, saturation_points=saturation_points)
visualizer_pred_hellinger.plot_graph_dashboard(slope_idx_init=6)

interactive(children=(Dropdown(description='Circuit:', options=(('varQC_linear_2r_r0', 0), ('varQC_linear_4r_r…